In [5]:
import numpy as np
import requests
import pandas
import json
import unicodeblock.blocks
import os
# Wikimedia requires a descriptive User-Agent. Put your real contact info here.
HEADERS = {"User-Agent": "loanword-replication/1.0 (your_email@example.com)"}
os.makedirs("results", exist_ok=True)

In [6]:
# expected Unicode blocks for certain scripts
unicode_block_map = {
    "Latin" : ['BASIC_LATIN', 'LATIN_1_SUPPLEMENT', 'LATIN_EXTENDED_LETTER', 'LATIN_EXTENDED_A', 'LATIN_EXTENDED_B',\
               'LATIN_EXTENDED_C'],
    "Greek" : ['GREEK'],
    "Cyrillic" : ['CYRILLIC', 'CYRILLIC_SUPPLEMENTARY', 'CYRILLIC_EXTENDED_A', 'CYRILLIC_EXTENDED_B'],
    "Arabic" : ['ARABIC', 'ARABIC_SUPPLEMENT', 'ARABIC_PRESENTATION_FORMS_A', 'ARABIC_PRESENTATION_FORMS_B'],
    "Devanagari" : ['DEVANAGARI', 'VEDIC_EXTENSIONS', 'DEVANAGARI_EXTENDED'],
    "Bengali" : ['BENGALI'],
    "Gurmukhi" : ['GURMUKHI'],
    "Tamil" : ['TAMIL'],
    "Telugu" : ['TELUGU'],
    "Malayalam" : ['MALAYALAM'],
    "Myanmar" : ['MYANMAR', 'MYANMAR_EXTENDED_A'],
    "Chinese" : ['CJK_RADICALS_SUPPLEMENT', 'CJK_SYMBOLS_AND_PUNCTUATION', 'CJK_STROKES',\
                 'ENCLOSED_CJK_LETTERS_AND_MONTHS', 'CJK_COMPATIBILITY', 'CJK_UNIFIED_IDEOGRAPHS_EXTENSION_A',\
                 'CJK_UNIFIED_IDEOGRAPHS', 'CJK_COMPATIBILITY_IDEOGRAPHS', 'CJK_COMPATIBILITY_FORMS',\
                 'CJK_UNIFIED_IDEOGRAPHS_EXTENSION_B', 'CJK_UNIFIED_IDEOGRAPHS_EXTENSION_C',\
                 'CJK_UNIFIED_IDEOGRAPHS_EXTENSION_D', 'CJK_COMPATIBILITY_IDEOGRAPHS_SUPPLEMENT']
}

In [3]:
"""def get_all_lemmas(language, expected_unicode, invalid=["Unsupported titles/"]):
    title = f"Category:{language}_lemmas"
    params = {
        'action': 'query',
        'format': 'json',
        'prop': 'extracts',
        'exintro': True,
        'explaintext': True,
    }

    url = f"https://en.wiktionary.org/w/api.php?action=query&list=categorymembers&cmtitle={title}&cmlimit=max"
    
    lemmas = []
    while(True):
        r = requests.get(url,params)
        try:
            cmcontinue = r.json()['continue']['cmcontinue']
            for cmember in r.json()['query']['categorymembers']:
                if len(cmember['title']) > 1 and \
                    not cmember['title'].startswith('-') and \
                    not cmember['title'].endswith('-') and \
                    not any(inv in cmember['title'] for inv in invalid) and \
                    not any(c.isdigit() for c in cmember['title']) and \
                    not cmember['title'].isupper() and \
                    unicodeblock.blocks.of(cmember['title'][0]) not in ['BASIC_PUNCTUATION'] and \
                    [unicodeblock.blocks.of(c) in unicode_block_map[expected_unicode] for c in cmember['title']].count(True) > len(cmember['title'])*.8:
                    lemmas.append(cmember['title'])
                    if len(lemmas) % 1000 == 0:
                        print(f"Got {len(lemmas)}")
            url = url.split("&cmcontinue")[0]
            url+=f"&cmcontinue={cmcontinue}"
        except KeyError:
            break
            
    df = pandas.DataFrame(np.array(lemmas).reshape(-1,1), columns=["word"])
    
    print(f"{language} finished ({len(lemmas)} total)\n")
    
    df.to_csv(f'results/{language}-AllLemmas.csv', index=False)"""

In [7]:
import time

def get_all_lemmas(language, expected_unicode, invalid=["Unsupported titles/"]):
    title = f"Category:{language}_lemmas"
    params = {
        'action': 'query',
        'format': 'json',
        'list': 'categorymembers',
        'cmtitle': title,
        'cmlimit': 'max',
    }

    lemmas = []
    retries = 0
    while True:
        r = requests.get("https://en.wiktionary.org/w/api.php",
                         params=params, headers=HEADERS, timeout=60)
        try:
            r.raise_for_status()
            data = r.json()
        except (requests.HTTPError, ValueError):
            retries += 1
            print(f"Bad response (status {r.status_code}): {r.text[:150]!r}")
            if retries > 5:
                raise
            time.sleep(30)
            continue
        retries = 0

        # process this page BEFORE checking whether there is a next one
        for cmember in data.get('query', {}).get('categorymembers', []):
            t = cmember['title']
            if len(t) > 1 and \
                not t.startswith('-') and \
                not t.endswith('-') and \
                not any(inv in t for inv in invalid) and \
                not any(c.isdigit() for c in t) and \
                not t.isupper() and \
                unicodeblock.blocks.of(t[0]) not in ['BASIC_PUNCTUATION'] and \
                [unicodeblock.blocks.of(c) in unicode_block_map[expected_unicode] for c in t].count(True) > len(t)*.8:
                lemmas.append(t)
                if len(lemmas) % 1000 == 0:
                    print(f"Got {len(lemmas)}")

        if 'continue' not in data:
            break
        params['cmcontinue'] = data['continue']['cmcontinue']
        time.sleep(0.5)

    df = pandas.DataFrame(np.array(lemmas).reshape(-1,1), columns=["word"])
    print(f"{language} finished ({len(lemmas)} total)\n")
    df.to_csv(f'results/{language}-AllLemmas.csv', index=False)


In [8]:
with open("../language-pairs.json", 'r') as f:
    pairs = json.loads(f.read())

for pair in pairs:
    print(pair)
    L2 = pairs[pair]['source']['name']
    L2_unicode = pairs[pair]['source']['unicode']
    
    if os.path.exists("results/{}-AllLemmas.csv".format(L2)):
        overwrite = input("{}-AllLemmas.csv exists. Overwrite existing file? (y/n) ".format(L2))
        if overwrite == "y":
            get_all_lemmas(L2, L2_unicode)
    else:
        get_all_lemmas(L2, L2_unicode)

Hindi-Persian
Persian-AllLemmas.csv exists. Overwrite existing file? (y/n) n
English-French
French-AllLemmas.csv exists. Overwrite existing file? (y/n) n
Finnish-Swedish
Swedish-AllLemmas.csv exists. Overwrite existing file? (y/n) n
Kazakh-Russian
Russian-AllLemmas.csv exists. Overwrite existing file? (y/n) n
Azerbaijani-Arabic
Arabic-AllLemmas.csv exists. Overwrite existing file? (y/n) n
Chinese-English
English-AllLemmas.csv exists. Overwrite existing file? (y/n) n
German-French
French-AllLemmas.csv exists. Overwrite existing file? (y/n) n
English-German
German-AllLemmas.csv exists. Overwrite existing file? (y/n) n
Romanian-Hungarian
Hungarian-AllLemmas.csv exists. Overwrite existing file? (y/n) n
Polish-French
French-AllLemmas.csv exists. Overwrite existing file? (y/n) n
Indonesian-Dutch
Got 1000
Got 2000
Got 3000
Got 4000
Got 5000
Got 6000
Got 7000
Got 8000
Got 9000
Got 10000
Got 11000
Got 12000
Got 13000
Got 14000
Got 15000
Got 16000
Got 17000
Got 18000
Got 19000
Got 20000
Got 2100